A News-Based Stock Prediction

## 1) Install dependencies

In [ ]:
!pip -q install -U finnhub-python yfinance pandas numpy tqdm regex
!pip -q install -U "transformers>=4.41.0" accelerate peft
!pip -q install -U bitsandbytes || true


## 2) Keys & environment

In [ ]:
import os

# Finnhub news API
FINNHUB_KEY = os.environ.get("FINNHUB_KEY", "")

HF_TOKEN = os.environ.get("HF_TOKEN", "")


## 3) Imports & device

In [ ]:
import datetime as dt
import time
import re
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import finnhub
import yfinance as yf

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


## 4) Finnhub client

In [ ]:
finnhub_client = finnhub.Client(api_key=FINNHUB_KEY)


## 5) Fetch & clean news (Finnhub)

In [ ]:
def get_company_news(symbol: str, weeks: int = 3, top_k: int = 15) -> List[Dict]:
    """Fetch recent company news via Finnhub and apply light cleaning."""
    to_date = dt.date.today()
    from_date = to_date - dt.timedelta(days=7 * weeks)

    raw = finnhub_client.company_news(
        symbol,
        _from=from_date.strftime("%Y-%m-%d"),
        to=to_date.strftime("%Y-%m-%d")
    ) or []

    seen = set()
    cleaned = []
    for x in raw:
        headline = (x.get("headline") or "").strip()
        summary  = (x.get("summary") or "").strip()
        if len(headline) < 8 or len(summary) < 40:
            continue
        key = headline.lower()
        if key in seen:
            continue
        seen.add(key)
        cleaned.append({
            "headline": headline,
            "summary": summary,
        })

    # Finnhub typically returns recent-first; keep Top-K
    return cleaned[:top_k]


## 6) Price window context

In [ ]:
def get_price_window(symbol: str, weeks: int = 3) -> Optional[Dict]:
    """Get a simple recent price move summary."""
    end = dt.date.today()
    start = end - dt.timedelta(days=7 * weeks + 7)  # buffer days

    df = yf.download(symbol, start=start, end=end + dt.timedelta(days=1), progress=False)
    if df is None or df.empty:
        return None
    df = df.dropna()
    if df.empty:
        return None

    p0 = float(df["Close"].iloc[0])
    p1 = float(df["Close"].iloc[-1])
    direction = "increased" if p1 >= p0 else "decreased"
    return {
        "start": str(df.index[0].date()),
        "end": str(df.index[-1].date()),
        "p0": p0,
        "p1": p1,
        "direction": direction,
    }


## 7) Build a FinGPT-Forecaster-style prompt

In [ ]:
SYSTEM_PROMPT = (
"You are a seasoned stock market analyst. "
"Your task is to list the positive developments and potential concerns for companies "
"based on relevant news from the past weeks, then provide an analysis and prediction "
"for the companies' stock price movement for the upcoming week.\n\n"
"[Positive Developments]:\n1. ...\n\n"
"[Potential Concerns]:\n1. ...\n\n"
"[Prediction & Analysis]:\nPrediction: (Up/Down + range)\nAnalysis: ...\n"
)

def build_prompt(symbol: str, weeks: int = 3, top_k_news: int = 12) -> str:
    price = get_price_window(symbol, weeks=weeks)
    news = get_company_news(symbol, weeks=weeks, top_k=top_k_news)

    if price:
        intro = (
            f"From {price['start']} to {price['end']}, {symbol}'s stock price {price['direction']} "
            f"from {price['p0']:.2f} to {price['p1']:.2f}."
        )
    else:
        intro = f"Recent price move for {symbol} is unavailable due to missing price data."

    news_block = ""
    if not news:
        news_block = "[Headline]: (No news retrieved)\n[Summary]: (No summary)\n"
    else:
        for n in news:
            news_block += f"[Headline]: {n['headline']}\n[Summary]: {n['summary']}\n\n"

    user_prompt = f"""[Company & Price Movement]
{intro}

[Recent News]
{news_block}

Based on the above, list 2-4 Positive Developments and 2-4 Potential Concerns for {symbol},
then predict next week's stock movement (Up/Down + range) and provide concise reasoning.
"""

    return SYSTEM_PROMPT + "\n" + user_prompt


## 8) Load model (choose ONE: merged model OR base+LoRA)

In [ ]:
USE_LORA = True


# (most common in GR5398): base model + LoRA adapter
BASE_MODEL_ID = "meta-llama/Llama-3.1-8B"  
LORA_ADAPTER_ID = "./lora_checkpoint" 

# Generation config
MAX_INPUT_LEN = 4096
MAX_NEW_TOKENS = 400
DO_SAMPLE = False
TEMPERATURE = 0.7
TOP_P = 0.9


In [ ]:
def load_merged(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or None, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=HF_TOKEN or None,
        device_map="auto" if DEVICE == "cuda" else None,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    mdl.eval()
    return mdl, tok

def load_base_plus_lora(base_id: str, adapter_id: str):
    tok = AutoTokenizer.from_pretrained(base_id, token=HF_TOKEN or None, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    base = AutoModelForCausalLM.from_pretrained(
        base_id,
        token=HF_TOKEN or None,
        device_map="auto" if DEVICE == "cuda" else None,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    mdl = PeftModel.from_pretrained(base, adapter_id)
    mdl.eval()
    return mdl, tok

if USE_LORA:
    model, tokenizer = load_base_plus_lora(BASE_MODEL_ID, LORA_ADAPTER_ID)
    print("Loaded base+LoRA:", BASE_MODEL_ID, "+", LORA_ADAPTER_ID)
else:
    model, tokenizer = load_merged(MERGED_MODEL_ID)
    print("Loaded merged model:", MERGED_MODEL_ID)


## 9) Generate forecast

In [ ]:
def generate_forecast(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.eos_token_id,
    )
    if DO_SAMPLE:
        gen_kwargs.update(dict(temperature=TEMPERATURE, top_p=TOP_P))

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    return tokenizer.decode(out[0], skip_special_tokens=True)

symbol = "AAPL"  # try: "MSFT", "NVDA", "TSLA"
prompt = build_prompt(symbol, weeks=3, top_k_news=12)

print("=== PROMPT (head) ===")
print(prompt[:1200])
print("\n=== OUTPUT ===")
out_text = generate_forecast(prompt)
print(out_text)


## 10) Parse prediction

In [ ]:
def parse_signal(text: str) -> Tuple[Optional[float], str]:
    t = text.lower()

    # Try to find an explicit prediction line
    pred_line = ""
    for line in text.splitlines():
        if "prediction" in line.lower():
            pred_line = line.lower()
            break
    s = pred_line if pred_line else t

    # Up by 2-3%
    m = re.search(r"up\s+by\s+(\d+(?:\.\d+)?)\s*[-~to]+\s*(\d+(?:\.\d+)?)%", s)
    if m:
        lo, hi = float(m.group(1))/100.0, float(m.group(2))/100.0
        return 0.5*(lo+hi), f"up {m.group(1)}-{m.group(2)}%"

    # Down by 1-2%
    m = re.search(r"down\s+by\s+(\d+(?:\.\d+)?)\s*[-~to]+\s*(\d+(?:\.\d+)?)%", s)
    if m:
        lo, hi = float(m.group(1))/100.0, float(m.group(2))/100.0
        return -0.5*(lo+hi), f"down {m.group(1)}-{m.group(2)}%"

    if "up" in s and "down" not in s:
        return 0.01, "up (weak)"
    if "down" in s and "up" not in s:
        return -0.01, "down (weak)"
    return None, "unparsed"

sig, label = parse_signal(out_text)
print("Parsed signal:", sig, "| label:", label)


## 11) Batch mode (multiple tickers)

In [ ]:
def forecast_many(symbols: List[str], weeks: int = 3, top_k_news: int = 10, sleep_s: float = 0.25) -> pd.DataFrame:
    rows = []
    for sym in tqdm(symbols):
        try:
            p = build_prompt(sym, weeks=weeks, top_k_news=top_k_news)
            txt = generate_forecast(p)
            sig, label = parse_signal(txt)
            rows.append({
                "ticker": sym,
                "signal": sig,
                "label": label,
                "output_head": txt[:1200]
            })
        except Exception as e:
            rows.append({"ticker": sym, "signal": None, "label": f"error: {e}", "output_head": ""})
        time.sleep(sleep_s)
    return pd.DataFrame(rows)

tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL"]
df = forecast_many(tickers, weeks=3, top_k_news=10)
df


## 12) Save outputs

In [ ]:
df.to_csv("forecaster_signals.csv", index=False)
print("Saved: forecaster_signals.csv")